# RAILGUN+ — Notebook 2: Full Evaluation

Compares four methods across agent counts and produces paper-style tables + plots:
- **expert** (PIBT oracle) — upper reference
- **pibt_only** (pure PIBT, no net) — shows what the network adds
- **greedy** (baseline RAILGUN) — the deadlock collapse
- **corrected** (net + PIBT corrector) — the contribution

Metrics: CSR, normalized SoC (SoC / lower-bound), makespan.

*Note: SCRIMP/DCC/etc. are NOT run here — that requires POGEMA's official harness (see `pogema_benchmark_stub` in the repo). These four are the runnable, honest baselines for the deadlock claim.*

## 1. Mount, clone, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT='/content/drive/MyDrive/railgun-plus'
CKPT_DIR=f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR=f'{DRIVE_ROOT}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
REPO_URL='https://github.com/tay805/railgun-plus.git'
%cd /content
!rm -rf railgun-plus
!git clone $REPO_URL
%cd /content/railgun-plus
!pip install -q pogema pyyaml tqdm
sys.path.insert(0,'/content/railgun-plus/src')

## 2. Load trained model

In [ ]:
import torch
from railgun_plus.models import RailgunUNet
device='cuda' if torch.cuda.is_available() else 'cpu'
model=RailgunUNet(6,5,base=64).to(device)
ckpt=torch.load(f'{CKPT_DIR}/best.pt',map_location=device)
model.load_state_dict(ckpt['model']);model.eval()
print('loaded epoch',ckpt['epoch'],'on',device)

## 3. Build held-out test sets

In [ ]:
import os, pickle
from railgun_plus.data.generate import generate_with_pogema

AGENT_COUNTS = [16, 32, 64, 96, 128]
N_PER = 20
TEST_PATH = f'{RESULTS_DIR}/test_sets.pkl'

# load whatever exists, then fill in only the missing agent counts
test_sets = {}
if os.path.exists(TEST_PATH):
    with open(TEST_PATH, 'rb') as f:
        test_sets = pickle.load(f)

for k in AGENT_COUNTS:
    if k in test_sets and len(test_sets[k]) > 0:
        print(k, 'agents: already have', len(test_sets[k]), '(skipped)')
        continue
    test_sets[k] = generate_with_pogema(N_PER, 32, 0.2, k, seed=1000 + k)
    print(k, 'agents:', len(test_sets[k]), 'instances (generated)')
    with open(TEST_PATH, 'wb') as f:
        pickle.dump(test_sets, f)   # checkpoint after each count

print('test sets ready:', {k: len(v) for k, v in test_sets.items()})

## 4. Run the full sweep (all four methods)
This runs expert, pibt_only, greedy, corrected on every agent count.

In [ ]:
from railgun_plus.eval.harness import run_sweep, sweep_to_table, plot_sweep
sweep = run_sweep(model, test_sets, device=device)

## 5. Paper-style results table (saved to Drive as CSV)

In [ ]:
rows = sweep_to_table(sweep, out_csv=f'{RESULTS_DIR}/results_table.csv')
# pretty-print as a dataframe
import pandas as pd
df = pd.DataFrame(rows)
display(df.pivot(index='agents', columns='method', values='csr'))
print('\nCSR table above. Full table (CSR/SoC-ratio/makespan) saved to Drive.')
df

## 6. Plot: CSR vs agents (the headline)

In [ ]:
plot_sweep(sweep, metric='csr',
           title='CSR vs agents: deadlock collapse vs PIBT correction',
           savepath=f'{RESULTS_DIR}/csr_all_methods.png')

## 7. Plot: normalized SoC (solution quality)
SoC / lower-bound. 1.0 = optimal. Only solved instances count, so read it next to CSR.

In [ ]:
plot_sweep(sweep, metric='avg_soc_ratio_solved',
           title='Solution quality (SoC / lower-bound; lower=better)',
           savepath=f'{RESULTS_DIR}/soc_ratio_all_methods.png')

## 8. Plot: makespan (latest arrival; paper Table I reports this)

In [ ]:
plot_sweep(sweep, metric='avg_makespan_solved',
           title='Makespan vs agents (solved instances)',
           savepath=f'{RESULTS_DIR}/makespan_all_methods.png')

## 9. (Later) SCRIMP / DCC / MAPF-GPT comparison
Run this only when the scaled-up model is solid. It points you to the right approach.

In [ ]:
from railgun_plus.eval.harness import pogema_benchmark_stub
# pogema_benchmark_stub()  # uncomment to read the roadmap docstring / NotImplementedError